<a href="https://colab.research.google.com/github/callsourav1979-personal/Assignments_HAAI-/blob/main/CV__Sorting_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!ls -lh /content/

total 4.0K
drwxr-xr-x 1 root root 4.0K Aug 18 13:44 sample_data


In [1]:
from pathlib import Path
cv_folder = Path("/content/cvs")
cv_folder.mkdir(exist_ok = True)

print(cv_folder)

/content/cvs


In [2]:
from pathlib import Path
jd_folder = Path("/content/jds")
jd_folder.mkdir(exist_ok = True)

In [ ]:
!mv /content/Resume1.pdf /content/cvs/
!mv /content/Resume2.pdf /content/cvs/
!mv /content/Resume3.docx /content/cvs/


In [ ]:
import torch
import sys

print("Python version:" , sys.version)
print("PyTorch version:" , torch.__version__)
print("CUDA available:" , torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:" , torch.version.cuda)
    print("GPU device name:" , torch.cuda.get_device_name(0))
    print("GPU Memory:" , round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),"GB")
else:
  print("Running on CPU")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch version: 2.11.0+cpu
CUDA available: False
Running on CPU


In [3]:
!pip install -q pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 23.8 MB/s eta 0:00:00


In [4]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils
!pip install -q pytesseract pdf2image

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 118422 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up poppler-utils (22.02.0-2ubuntu0.13) ...
Processing triggers for man-db (2.10.2-1) ...


In [5]:
import sys
import pypdf
import docx

print("Python version:" , sys.version)
print("PyPdf version:" , pypdf.__version__)
print("python-docx version:" , docx.__version__)
#

Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyPdf version: 6.16.1
python-docx version: 1.2.0


In [6]:
import pytesseract
from pdf2image import convert_from_path
from pathlib import Path

print("Python version:" , sys.version)
print("PyTesseract version:" , pytesseract.__version__)


Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTesseract version: 0.3.13


In [7]:
from pypdf import PdfReader
from docx import Document
from pathlib import Path

def extract_text_from_file(file_path):
    """
    Extract text from PDF or DOCX files.

      For PDF:
          Extracts texts from all pages.

      For DOCX:
          Extracts texts from normal paragraphs and tables.

      Parameters :
           file_path(str): Path to the PDF or DOCX file.
      Returns :
           str: Extracted text from the file.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = path.suffix.lower()
    #---------------------------------
    # PDF
    #---------------------------------
    if extension == '.pdf':
        reader = PdfReader(str(path))

        pages = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
              pages.append(text)
        return '\n'.join(pages).strip()
    #---------------------------------------
    # DOCX
    #---------------------------------------
    elif extension == '.docx':
        document = Document(str(path))

        sections = []
        # Extract normal paragraphs
        for paragraph in document.paragraphs:
            text =  paragraph.text.strip()
            if text:
              sections.append(text)

        # Extract tables
        for table in document.tables:
            sections.append("\n[TABLE ]")
            for row in table.rows:
                row_cells = []
                for cell in row.cells:
                    cell_text = cell.text.strip()
                    if cell_text:
                      row_cells.append(cell_text)
                #Combine cells in the same row
                if row_cells:
                   sections.append(" | ".join(row_cells))
            sections.append("[/TABLE]")
        #Combine paragraphs and tables
        extracted_text = "\n".join(sections).strip()

        return extracted_text

    else:
      raise ValueError(f"Unsupported file type: {extension}" "Only PDF and DOCX files are supported.")


In [8]:
from pdf2image import convert_from_path
import pytesseract
from pathlib import Path

def extract_text_from_pdf_ocr(file_path):
    """
    Extract text from scanned/image based PDF files using OCR

     Parameters :
           file_path(str): Path to the PDF file.
      Returns :
           str: OCR Extracted text from the file.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = path.suffix.lower()
    if extension != '.pdf':
      raise ValueError(f"This function supports PDF files only.")

    # Convert PDF pages into images
    pages = convert_from_path(str(path), dpi=300)

    extracted_pages = []

    # Process each page
    for page_number , page_image in enumerate(pages, start =1):
        print(f"Processing page {page_number}/{len(pages)}")
        #Run OCR
        text = pytesseract.image_to_string(page_image,config="--psm 6")

        #Remove unnessary whitespace
        text = text.strip()

        #Store page seperately
        page_text = (f"\n---PAGE {page_number} ---\n" f"{(text)}")

    extracted_pages.append(page_text)

    # Combine all pages
    final_text = "\n".join(extracted_pages)

    return final_text.strip()

## **`Wrapper Program`**

In [9]:
from pathlib import Path

def extract_document_text(file_path):
  """
  Main document-extraction wrapper
  Automatically selects the appropriate extraction method based on the file type
  and available text.

    Parameters:
       file_path (str) : Path to the resume file

    Returns:
       str : Extracted text from the resume
  """
  path = Path(file_path)

  #1. Check whether the file exists
  if not path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")

  #2. Check supported file types
  extension = path.suffix.lower()
  if extension not in ['.pdf' , '.docx']:
    raise ValueError(f"Unsupported file type: {extension}" "Only PDF and DOCX files are supported.")

  #3 Handle PDF
  if extension == '.pdf':
    print(f"\nProcessng PDF: {path.name}")

    #First try normal PDF text extraction
    text = extract_text_from_file(path)

    #Check whether meaningful text was extracted
    if text and len(text.strip()) >= 100 :
      print("Text layer detected. Using standard PDF extraction.")
      return text.strip()
    if len(text.strip()) < 100 :
      print("Little or no text detected . Swtching to OCR ...")
      text = extract_text_from_pdf_ocr(path)
      return text.strip()

  #4 Handle DOCX
  elif extension == '.docx':
    print(f"\nProcessing DOCX: {path.name}")
    text = extract_text_from_file(path)
    return text.strip()

## **Extract Multiple CV's**

In [10]:
from pathlib import Path

def extract_multiple_cvs(cv_folder):
    """
    Extract text from all supported CV files in a folder.

    Supported formats:
      - PDF
      - DOCX

    Parameters:
       cv_folder (str) : Path to the folder containing CVs

    Returns:
       dict: Dictionary containing filename and extracted text
    """

    folder = Path(cv_folder)
    if not folder.exists():
        raise FileNotFoundError(f"CV Folder not found: {cv_folder}")

    if not folder.is_dir():
        raise ValueError(f"Path is not a directory: {cv_folder}")

    # Find all PDF and DOCX files
    cv_files = sorted(
                       [ file
                         for file in folder.iterdir()
                         if file.is_file() and file.suffix.lower() in [".pdf" , ".docx"]
                        ]
                      )
    if not cv_files:
      raise ValueError(f"No PDF or DOCX files found in :  {cv_folder}")

    cv_data = {}

    for cv_file in cv_files:
      print("="*60)
      print(f"Processing CV: {cv_file.name}")
      print("="*60)

      try:
        text = extract_document_text(cv_file)
        cv_data[cv_file.name] = text
        print(f"Characters Extracted: {len(text)}")
      except Exception as e:
        print(f"Error processing {cv_file.name}: {e}")
        cv_data[cv_file.name] = ""

    return cv_data

In [ ]:
from pathlib import Path
cv_folder = Path("/content/cvs")

for cv_file in sorted(cv_folder.iterdir()):
  if cv_file.is_file():
    text = extract_text_from_file(cv_file)
    print("\n" + "="*70)
    print("FILE:" , cv_file.name)
    print("Characters extracted :" , len(text))
    print("="*70)
    print(text[:2000])



# **Install Transformer & Model**

In [11]:
!pip install -q transformers torch accelerate


In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer=AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)
model=AutoModelForCausalLM.from_pretrained(model_name,torch_dtype=torch.float32,trust_remote_code=True,device_map="auto")

print("Model loaded Successfully")

## **READ JD from Folder**

In [ ]:
jd_folder ="/content/jds"
all_jd = extract_multiple_cvs(jd_folder)
print("Total JDs processed:" , len(all_jd))

In [45]:
print(all_jd.keys())

dict_keys(['JD-AI Engineer.pdf', 'JD-Senior Software Engineer.pdf', 'JD-Senior-Data-Analyst.pdf', 'JD-SofrwareEngineer.pdf', 'JD-SpecialEducator.pdf', 'JD-Technical_Project_Manager.pdf'])


In [ ]:
jd_text= all_jd["JD-SofrwareEngineer.pdf"]
print(jd_text[:10000])

# **JD to JSON Conversion PROMPT**

In [ ]:
jd_prompt = """
You are an expert recruitment assistant.

Your task is to analyze the following Job Description and convert the information explicitly stated in it into a structured JSON format.

JOB DESCRIPTION :
""" + jd_text + """

Extract the following information:
1. Job title
2. Required Skills
3. Minimum experience required
4. Educational requirements
5. Job responsibilities
6. Important keywords

Return ONLY vaid JSON in exactly this format:

{
  "job_title": "",
  "skills": [],
  "experience": [],
  "education": [],
  "responsibilities": []
}

IMPORTANT RULES:

1. Extract information ONLY from the CV. Do not use outside knowledge or assumptions.
2. Do not invent, infer, or assume any skill, experience, responsibility, education, or qualification.
3. Extract skills explicitly mentioned in the CV. Do not convert ordinary personal characteristics into technical skills.
4. Extract work experience only when work experience is explicitly mentioned. Do not convert statements such as "Fresher" into a number of years.
5. Extract education/qualifications exactly from the CV.
6. Extract responsibilities, duties, projects, or work activities only when they are explicitly mentioned.
7. Extract important keywords that are explicitly present in the CV.
8. Preserve the meaning and wording of the CV as closely as possible.
9. If information is not available, return an empty list [] for list fields and an empty string "" for candidate_name or job_title.
10. Do not output values such as "None specified", "Not specified", "Not mentioned", "Unknown", or similar text. Use [] or "" instead.
11. Do not include personal information such as date of birth, gender, father's name, address, phone number, or email in the JSON unless it is necessary for identifying the candidate name. These details are not relevant for job matching.
12. Do not treat personal characteristics such as "hardworking", "positive attitude", or "quick learner" as technical skills.
13. Return ONLY the JSON object. Do not provide explanations, comments, Markdown, or any text outside the JSON.
"""

messages = [
      {"role": "system", "content": "You are a precise recruitment assistant that extracts information from CV"},
      {"role": "user", "content": jd_prompt}
  ]

text= tokenizer.apply_chat_template(messages , tokenize=False , add_generation_prompt= True)

inputs = tokenizer(text, return_tensors="pt").to(model.device)


with torch.no_grad():
  outputs = model.generate(**inputs,max_new_tokens=500,do_sample=False)

jd_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)

print("LLM Response:" , jd_response.strip())


# **READ CV's from Folder**

In [62]:
cv_folder ="/content/cvs"
all_cvs = extract_multiple_cvs(cv_folder)
print("Total CVs processed:" , len(all_cvs))

Processing CV: Resume4.pdf

Processng PDF: Resume4.pdf
Little or no text detected . Swtching to OCR ...
Processing page 1/1
Characters Extracted: 869
Processing CV: data_analyst_resume_junior.pdf

Processng PDF: data_analyst_resume_junior.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 1612
Processing CV: data_analyst_resume_mid_level.pdf

Processng PDF: data_analyst_resume_mid_level.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 1920
Processing CV: data_analyst_resume_mid_level_v2.pdf

Processng PDF: data_analyst_resume_mid_level_v2.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 2436
Processing CV: technical_program_manager_cv.pdf

Processng PDF: technical_program_manager_cv.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 3249
Total CVs processed: 5


In [63]:
print(all_cvs.keys())

dict_keys(['Resume4.pdf', 'data_analyst_resume_junior.pdf', 'data_analyst_resume_mid_level.pdf', 'data_analyst_resume_mid_level_v2.pdf', 'technical_program_manager_cv.pdf'])


In [64]:
cv_text= all_cvs["technical_program_manager_cv.pdf"]
print(cv_text[:10000])

ALEX MORGAN, PMP
Principal Technical Program Manager — Cloud Infrastructure & AI
Systems
 alex.morgan@email.com  (555) 019-2834
 linkedin.com/in/alexmorgan-tpm
 Seattle, WA
Professional Profile
Results-driven Technical Program Manager with over 8 years of experience leading cross-functional engineering teams to
deliver complex, large-scale cloud infrastructure and machine learning products. Adept at bridging the gap between business
strategy and deep technical execution. Expert in managing multi-million dollar portfolios, mitigating architectural risks, and
optimizing Agile/Scrum lifecycles to accelerate time-to-market.
Core Competencies & Technical Skills Matrix
Domain / Core Expertise
Technologies, Toolsets & Frameworks
Cloud & Distributed Systems
AWS (EC2, S3, RDS, Lambda), Azure, Docker, Kubernetes, Microservices Architecture
Program & Agile Management
Jira Portfolio, Confluence, Asana, Scaled Agile Framework (SAFe), Scrum, Kanban
Data & AI Infrastructure
Python, SQL, Apache Spark

# **CV to JSON Conversion PROMPT**

In [67]:
cv_prompt = """
You are an expert recruitment assistant.

Your task is to analyze the following CV/Resume and convert the information explicitly stated in it into a structured JSON format.

JOB DESCRIPTION :
""" + cv_text + """

Extract the following information:
1. Candidate Name
2. Job title or professional title, if explicitly stated
3. Skills explicitly mentioned in the CV
4. Work experience explicitly mentioned in the CV
5. Education/qualifications explicitly mentioned in the CV
6. Job responsibilities , duties , projects , or work activities explicitly mentioned in the CV
6. Important keywords explicitly mentioned in the CV

Return ONLY vaid JSON in exactly this format:

{
  "candidate_name": "",
  "job_title": "",
  "skills": [],
  "experience": [],
  "education": [],
  "responsibilities": []
}

IMPORTANT RULES:

1. Extract information only from the CV. Do not use outside
   knowledge or assumptions.

2. Preserve the original meaning and wording of the Job Description
   as closely as possible.

3. Do NOT classify anything as required, preferred, or other at this stage.

4. "skills" must contain actual skills, abilities, knowledge, tools,
   technologies, software, programming languages, or competencies
   explicitly mentioned in the Job Description.

5. "experience" must contain explicitly stated experience requirements
   or experience statements.

6. "education" must contain explicitly stated educational qualifications,
   degrees, diplomas, certifications, or fields of study.

7. "responsibilities" must contain the actual duties and responsibilities
   stated in the Job Description.

8. Do NOT convert responsibilities into skills.

9. Do NOT convert education into skills.

10. Do NOT convert experience into skills.

11. Keep responsibilities as complete statements. Do not unnecessarily
    summarize or omit important responsibilities.

12. Extract important keywords explicitly present in the Job Description.

13. If information is not present, return an empty list [].

14. NEVER output "None specified", "Not specified", "Not mentioned",
    "Unknown", or similar text. Use [] instead.

15. Return ONLY the JSON object. Do not provide explanations,
    comments, markdown, or text outside the JSON.
"""

messages = [
      {"role": "system", "content": "You are a precise recruitment assistant that extracts information from CV"},
      {"role": "user", "content": cv_prompt}
  ]

text= tokenizer.apply_chat_template(messages , tokenize=False , add_generation_prompt= True)

inputs = tokenizer(text, return_tensors="pt").to(model.device)


with torch.no_grad():
  outputs = model.generate(**inputs,max_new_tokens=800,do_sample=False)

cv_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)

print("LLM Response:" , cv_response.strip())


LLM Response: ```json
{
  "candidate_name": "ALEX MORGAN",
  "job_title": "Principal Technical Program Manager — Cloud Infrastructure & AI",
  "skills": [
    "AWS (EC2, S3, RDS, Lambda)",
    "Azure",
    "Docker",
    "Kubernetes",
    "Microservices Architecture",
    "Jira Portfolio",
    "Confluence",
    "Asana",
    "Scaled Agile Framework (SAFe)",
    "Scrum",
    "Kanban",
    "Python",
    "SQL",
    "Apache Spark",
    "TensorFlow pipelines",
    "Data Warehousing (Snowflake)",
    "Risk Mitigation",
    "OKR Mapping",
    "Vendor Management",
    "Budgeting"
  ],
  "experience": [
    {
      "company": "NextGen Tech Solutions",
      "position": "Principal Technical Program Manager",
      "duration": "2023 – Present",
      "responsibilities": [
        "Directed the migration of legacy monolithic data suites to a distributed AWS Kubernetes microservices mesh, reducing infrastructure costs by 22% ($1.4M annualized savings).",
        "Spearheaded a cross-functional progra